# 02 — Demographics Assembly

Combines CB-level demographics (housing, population) from the 2020 Decennial Census with CBG-level Median Household Income (MHI) from ACS. Computes household and population shares, occupancy rates, allocates 2024 ACS projections to the CB level, and joins area data for density computation.

**Input Files:**
- CB Housing: `data/Demographics/Housing/DECENNIALPL2020.H1-Data_CB.csv` (5 states: CA, GA, IL, NY, TX)
- CB Housing (MI): `data/MI/DECENNIALPL2020.H1-Data.csv`
- CB Population: `data/Demographics/Population/DECENNIALDHC2020.P1-Data.csv` (5 states: CA, GA, IL, NY, TX)
- CB Population (MI): `data/MI/DECENNIALDHC2020.P1-Data.csv`
- CBG Housing (Decennial 2020): `data/Demographics/Housing/DECENNIALPL2020.H1-Data.csv`
- CBG HU Estimates (ACS 2020): `data/Demographics/Housing/ACSDT5Y2020.B25001-Data.csv`
- CBG HU Estimates (ACS 2024): `data/Demographics/Housing/ACSDT5Y2024.B25001-Data.csv`
- CBG Pop Estimates (ACS 2024): `data/Demographics/Population/ACSDT5Y2024.B01003-Data.csv`
- MHI (ACS 2020): `data/Demographics/Median HH Income/ACSDT5Y2020.B19013-Data.csv`
- MHI (ACS 2024): `data/Demographics/Median HH Income/ACSDT5Y2024.B19013-Data.csv`
- Area Data: `output/area_sq_km.csv` (from 01_shapefile_processing)

**Output:** `output/cb_demographics.parquet`

**Prerequisite:** `01_shapefile_processing.ipynb` must be run first to produce `output/area_sq_km.csv`.

**Note:** CB-level data covers 6 states (CA, GA, IL, MI, NY, TX). CBG-level data is nationwide — unmatched CBGs outside these states are expected and filtered out during joins.

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

# --- File Paths ---
DATA_DIR = '../../data/Demographics'
MI_DATA_DIR = '../../data/MI'
HOUSING_DIR = os.path.join(DATA_DIR, 'Housing')
POP_DIR = os.path.join(DATA_DIR, 'Population')
MHI_DIR = os.path.join(DATA_DIR, 'Median HH Income')
OUTPUT_DIR = '../../output'

# CB-Level (2020 Decennial Census — 5 states: CA, GA, IL, NY, TX)
CB_HOUSING_FILE = os.path.join(HOUSING_DIR, 'DECENNIALPL2020.H1-Data_CB.csv')
CB_POP_FILE = os.path.join(POP_DIR, 'DECENNIALDHC2020.P1-Data.csv')

# CB-Level (MI — from data/MI/)
MI_CB_HOUSING_FILE = os.path.join(MI_DATA_DIR, 'DECENNIALPL2020.H1-Data.csv')
MI_CB_POP_FILE = os.path.join(MI_DATA_DIR, 'DECENNIALDHC2020.P1-Data.csv')

# CBG-Level (nationwide)
CBG_HOUSING_DECENNIAL = os.path.join(HOUSING_DIR, 'DECENNIALPL2020.H1-Data.csv')
CBG_HU_ACS_2020 = os.path.join(HOUSING_DIR, 'ACSDT5Y2020.B25001-Data.csv')
CBG_HU_ACS_2024 = os.path.join(HOUSING_DIR, 'ACSDT5Y2024.B25001-Data.csv')
CBG_POP_ACS_2024 = os.path.join(POP_DIR, 'ACSDT5Y2024.B01003-Data.csv')

# MHI
MHI_2020_FILE = os.path.join(MHI_DIR, 'ACSDT5Y2020.B19013-Data.csv')
MHI_2024_FILE = os.path.join(MHI_DIR, 'ACSDT5Y2024.B19013-Data.csv')

# Area data from Phase 1 (01_shapefile_processing)
AREA_CSV = os.path.join(OUTPUT_DIR, 'area_sq_km.csv')

# 6-state FIPS codes
TARGET_STATES = {'06', '13', '17', '26', '36', '48'}
STATE_NAMES = {'06': 'CA', '13': 'GA', '17': 'IL', '26': 'MI', '36': 'NY', '48': 'TX'}

print("File paths configured.")

## 2. Load CB-Level Demographics (2020 Decennial Census)

Census Block (CB) level data from the 2020 Decennial Census provides foundational housing and population counts. These are **full enumeration** counts (not estimates), available for the 6 project states: CA, GA, IL, MI, NY, TX.

- **Housing:** Total HU, Occupied HU (Households), Vacant HU
- **Population:** Total population

The 5 original states (CA, GA, IL, NY, TX) are loaded from the pre-concatenated files, then MI is loaded separately from `data/MI/` and appended.

In [ ]:
### Load CB Housing (DECENNIALPL2020.H1 — Census Block level, 5 states)
cb_housing = pd.read_csv(CB_HOUSING_FILE, skiprows=[1], encoding='utf-8-sig', dtype=str)
cb_housing = cb_housing[[c for c in cb_housing.columns if not c.startswith('Unnamed')]]
cb_housing['cb_fips'] = cb_housing['GEO_ID'].str.split('US').str[1]

for col in ['H1_001N', 'H1_002N', 'H1_003N']:
    cb_housing[col] = pd.to_numeric(cb_housing[col], errors='coerce').fillna(0).astype(int)

cb_housing = cb_housing.rename(columns={
    'H1_001N': 'total_housing_units',
    'H1_002N': 'occupied_housing_units',
    'H1_003N': 'vacant_housing_units'
})[['cb_fips', 'total_housing_units', 'occupied_housing_units', 'vacant_housing_units']]

print(f"CB Housing (5 states): {len(cb_housing):,} records")

### Load MI CB Housing from data/MI/
mi_cb_housing = pd.read_csv(MI_CB_HOUSING_FILE, skiprows=[1], encoding='utf-8-sig', dtype=str)
mi_cb_housing = mi_cb_housing[[c for c in mi_cb_housing.columns if not c.startswith('Unnamed')]]
mi_cb_housing['cb_fips'] = mi_cb_housing['GEO_ID'].str.split('US').str[1]

for col in ['H1_001N', 'H1_002N', 'H1_003N']:
    mi_cb_housing[col] = pd.to_numeric(mi_cb_housing[col], errors='coerce').fillna(0).astype(int)

mi_cb_housing = mi_cb_housing.rename(columns={
    'H1_001N': 'total_housing_units',
    'H1_002N': 'occupied_housing_units',
    'H1_003N': 'vacant_housing_units'
})[['cb_fips', 'total_housing_units', 'occupied_housing_units', 'vacant_housing_units']]

print(f"CB Housing (MI):       {len(mi_cb_housing):,} records")

### Concatenate
cb_housing = pd.concat([cb_housing, mi_cb_housing], ignore_index=True)
del mi_cb_housing
print(f"CB Housing (6 states): {len(cb_housing):,} records")

### Load CB Population (DECENNIALDHC2020.P1 — Census Block level, 5 states)
cb_pop = pd.read_csv(CB_POP_FILE, skiprows=[1], encoding='utf-8-sig', dtype=str)
cb_pop = cb_pop[[c for c in cb_pop.columns if not c.startswith('Unnamed')]]
cb_pop['cb_fips'] = cb_pop['GEO_ID'].str.split('US').str[1]
cb_pop['total_population'] = pd.to_numeric(cb_pop['P1_001N'], errors='coerce').fillna(0).astype(int)
cb_pop = cb_pop[['cb_fips', 'total_population']]

print(f"CB Population (5 states): {len(cb_pop):,} records")

### Load MI CB Population from data/MI/
mi_cb_pop = pd.read_csv(MI_CB_POP_FILE, skiprows=[1], encoding='utf-8-sig', dtype=str)
mi_cb_pop = mi_cb_pop[[c for c in mi_cb_pop.columns if not c.startswith('Unnamed')]]
mi_cb_pop['cb_fips'] = mi_cb_pop['GEO_ID'].str.split('US').str[1]
mi_cb_pop['total_population'] = pd.to_numeric(mi_cb_pop['P1_001N'], errors='coerce').fillna(0).astype(int)
mi_cb_pop = mi_cb_pop[['cb_fips', 'total_population']]

print(f"CB Population (MI):       {len(mi_cb_pop):,} records")

### Concatenate
cb_pop = pd.concat([cb_pop, mi_cb_pop], ignore_index=True)
del mi_cb_pop
print(f"CB Population (6 states): {len(cb_pop):,} records")

### Merge Housing + Population at CB Level
cb_df = cb_housing.merge(cb_pop, on='cb_fips', how='inner')
cb_df['cbg_fips'] = cb_df['cb_fips'].str[:12]

print(f"\nMerged CB Demographics: {len(cb_df):,} records")
print(f"CB FIPS lengths: {cb_df['cb_fips'].str.len().value_counts().to_dict()}")
print(f"States: {sorted(cb_df['cb_fips'].str[:2].unique())}")
print(f"Unique CBGs: {cb_df['cbg_fips'].nunique():,}")
cb_df.head()

## 3. Load CBG-Level Reference Data

CBG-level data is loaded from multiple sources:
1. **2020 Decennial Census Housing (CBG)** — Total HU, Occupied HU, Vacant HU → Used for **occupancy rate** computation
2. **ACS 2020 Housing Units (CBG)** — Total HU estimate → Reference baseline
3. **ACS 2024 Housing Units (CBG)** — Total HU estimate → Used as **HU projection** for rolldown
4. **ACS 2024 Population (CBG)** — Total pop estimate → Used as **population projection** for rolldown

All CBG files are nationwide (~242K CBGs). Only CBGs matching our 6-state CB data will be used in the final output.

In [3]:
def load_census_cbg(filepath, value_cols, rename_map):
    """Load a CBG-level Census/ACS file, extract FIPS, and rename columns."""
    df = pd.read_csv(filepath, skiprows=[1], encoding='utf-8-sig', dtype=str)
    df = df[[c for c in df.columns if not c.startswith('Unnamed')]]
    df['cbg_fips'] = df['GEO_ID'].str.split('US').str[1]
    for col in value_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.rename(columns=rename_map)
    return df[['cbg_fips'] + list(rename_map.values())]

### CBG Housing — 2020 Decennial Census (for occupancy rate)
cbg_housing_dec = load_census_cbg(
    CBG_HOUSING_DECENNIAL,
    value_cols=['H1_001N', 'H1_002N', 'H1_003N'],
    rename_map={'H1_001N': 'cbg_total_hu_2020', 'H1_002N': 'cbg_hh_2020', 'H1_003N': 'cbg_vacant_2020'}
)
print(f"CBG Housing (Decennial 2020): {len(cbg_housing_dec):,} CBGs")

### CBG Housing Units — ACS 2020 (reference baseline)
cbg_hu_acs_2020 = load_census_cbg(
    CBG_HU_ACS_2020,
    value_cols=['B25001_001E'],
    rename_map={'B25001_001E': 'cbg_hu_acs_2020'}
)
print(f"CBG HU ACS 2020: {len(cbg_hu_acs_2020):,} CBGs")

### CBG Housing Units — ACS 2024 (projection source)
cbg_hu_acs_2024 = load_census_cbg(
    CBG_HU_ACS_2024,
    value_cols=['B25001_001E'],
    rename_map={'B25001_001E': 'cbg_hu_proj_2024'}
)
print(f"CBG HU ACS 2024: {len(cbg_hu_acs_2024):,} CBGs")

### CBG Population — ACS 2024 (projection source)
cbg_pop_acs_2024 = load_census_cbg(
    CBG_POP_ACS_2024,
    value_cols=['B01003_001E'],
    rename_map={'B01003_001E': 'cbg_pop_proj_2024'}
)
print(f"CBG Pop ACS 2024: {len(cbg_pop_acs_2024):,} CBGs")

CBG Housing (Decennial 2020): 242,333 CBGs
CBG HU ACS 2020: 242,333 CBGs
CBG HU ACS 2024: 242,297 CBGs
CBG Pop ACS 2024: 242,297 CBGs


## 4. Load Median Household Income (MHI)

MHI is reported at the **CBG level** (12-digit FIPS). Both 2020 and 2024 ACS 5-Year estimates are loaded.

- Values coded as `-` or suppressed by the Census Bureau become NaN after numeric conversion.
- Margin of Error (MOE) is retained for flagging low-confidence estimates.

In [4]:
### Load MHI 2020 (ACS 5-Year)
mhi_2020 = load_census_cbg(
    MHI_2020_FILE,
    value_cols=['B19013_001E', 'B19013_001M'],
    rename_map={'B19013_001E': 'mhi_2020', 'B19013_001M': 'mhi_moe_2020'}
)
print(f"MHI 2020: {len(mhi_2020):,} CBGs | Non-null MHI: {mhi_2020['mhi_2020'].notna().sum():,}")

### Load MHI 2024 (ACS 5-Year)
mhi_2024 = load_census_cbg(
    MHI_2024_FILE,
    value_cols=['B19013_001E', 'B19013_001M'],
    rename_map={'B19013_001E': 'mhi_2024', 'B19013_001M': 'mhi_moe_2024'}
)
print(f"MHI 2024: {len(mhi_2024):,} CBGs | Non-null MHI: {mhi_2024['mhi_2024'].notna().sum():,}")

### Merge both MHI vintages
mhi = mhi_2020.merge(mhi_2024, on='cbg_fips', how='outer')
print(f"\nCombined MHI table: {len(mhi):,} CBGs")
mhi.head()

MHI 2020: 242,333 CBGs | Non-null MHI: 225,286
MHI 2024: 242,297 CBGs | Non-null MHI: 220,201

Combined MHI table: 245,052 CBGs


,cbg_fips,mhi_2020,mhi_moe_2020,mhi_2024,mhi_moe_2024
0,010010201001,"39,167.00","20,140.00","42,750.00","16,123.00"
1,010010201002,"70,699.00","11,633.00","64,510.00","26,923.00"
2,010010202001,"39,750.00","20,003.00","48,750.00","35,352.00"
3,010010202002,"50,221.00","3,210.00","61,725.00","8,895.00"
4,010010203001,"67,813.00","11,351.00","65,132.00","20,268.00"


## 5. Assign MHI to Census Blocks (Uniform Rolldown)

MHI is a **median** — it cannot be split or weight-averaged across sub-geographies. The correct approach is **uniform assignment**: every CB within a CBG inherits the same MHI value from its parent CBG.

```
MHI_cb = MHI_cbg    (uniform — median is non-additive)
Join key: cb_fips[:12] == cbg_fips
```

In [5]:
### Left-join MHI to CB data on cbg_fips (uniform assignment)
print(f"CB records before MHI join: {len(cb_df):,}")

cb_df = cb_df.merge(mhi, on='cbg_fips', how='left')

print(f"CB records after MHI join:  {len(cb_df):,}")

mhi_match_2020 = cb_df['mhi_2020'].notna().sum()
mhi_match_2024 = cb_df['mhi_2024'].notna().sum()
print(f"\nMHI 2020 matched: {mhi_match_2020:,} / {len(cb_df):,} ({mhi_match_2020/len(cb_df)*100:.1f}%)")
print(f"MHI 2024 matched: {mhi_match_2024:,} / {len(cb_df):,} ({mhi_match_2024/len(cb_df)*100:.1f}%)")
cb_df[['cb_fips', 'cbg_fips', 'occupied_housing_units', 'total_population', 'mhi_2020', 'mhi_2024']].head(10)

CB records before MHI join: 2,079,994
CB records after MHI join:  2,079,994

MHI 2020 matched: 1,944,606 / 2,079,994 (93.5%)
MHI 2024 matched: 1,902,151 / 2,079,994 (91.4%)


,cb_fips,cbg_fips,occupied_housing_units,total_population,mhi_2020,mhi_2024
0,060014001001000,060014001001,0,0,"214,500.00",NaN
1,060014001001001,060014001001,0,0,"214,500.00",NaN
2,060014001001002,060014001001,0,0,"214,500.00",NaN
3,060014001001003,060014001001,0,0,"214,500.00",NaN
4,060014001001004,060014001001,0,0,"214,500.00",NaN
5,060014001001005,060014001001,0,0,"214,500.00",NaN
6,060014001001006,060014001001,0,0,"214,500.00",NaN
7,060014001001007,060014001001,0,0,"214,500.00",NaN
8,060014001001008,060014001001,0,0,"214,500.00",NaN
9,060014001001009,060014001001,2,11,"214,500.00",NaN


## 6. Compute Rolled-Up CBG Totals & CB Shares

**Critical rule:** CB-level counts come from the 2020 Decennial Census; CBG-level estimates come from ACS — different data products. The sum of CB values within a CBG will **not** match the ACS CBG estimate.

**Solution:** Always compute the denominator by rolling up CB values. This ensures shares within every CBG sum to exactly 1.0.

```
hh_share  = CB_occupied_HH / SUM(CB_occupied_HH within CBG)
pop_share = CB_population  / SUM(CB_population within CBG)
```

In [6]:
### Compute rolled-up CBG totals from CB data (use as denominators, NOT ACS CBG estimates)
cbg_rolled = cb_df.groupby('cbg_fips').agg(
    cbg_rolled_hh=('occupied_housing_units', 'sum'),
    cbg_rolled_hu=('total_housing_units', 'sum'),
    cbg_rolled_pop=('total_population', 'sum')
).reset_index()

print(f"Rolled-up CBG totals: {len(cbg_rolled):,} CBGs")

cb_df = cb_df.merge(cbg_rolled, on='cbg_fips', how='left')

### Compute HH share: CB occupied HH / rolled-up CBG total
cb_df['hh_share_in_cbg'] = np.where(
    cb_df['cbg_rolled_hh'] > 0,
    cb_df['occupied_housing_units'] / cb_df['cbg_rolled_hh'],
    0.0
)

### Compute Population share: CB pop / rolled-up CBG total
cb_df['pop_share_in_cbg'] = np.where(
    cb_df['cbg_rolled_pop'] > 0,
    cb_df['total_population'] / cb_df['cbg_rolled_pop'],
    0.0
)

### Validate: shares should sum to exactly 1.0 within each non-zero CBG
hh_sums = cb_df.groupby('cbg_fips')['hh_share_in_cbg'].sum()
hh_nonzero = hh_sums[hh_sums > 0]
hh_bad = ((hh_nonzero - 1.0).abs() > 1e-6).sum()
print(f"\nHH share validation — non-zero CBGs: {len(hh_nonzero):,} | Failed sum-to-1: {hh_bad}")

pop_sums = cb_df.groupby('cbg_fips')['pop_share_in_cbg'].sum()
pop_nonzero = pop_sums[pop_sums > 0]
pop_bad = ((pop_nonzero - 1.0).abs() > 1e-6).sum()
print(f"Pop share validation — non-zero CBGs: {len(pop_nonzero):,} | Failed sum-to-1: {pop_bad}")

### Zero-denominator CBGs (all CBs report 0 households/population)
zero_hh_cbgs = (hh_sums == 0).sum()
zero_pop_cbgs = (pop_sums == 0).sum()
print(f"\nZero-HH CBGs (all CBs have 0 occupied HU): {zero_hh_cbgs:,}")
print(f"Zero-Pop CBGs (all CBs have 0 population): {zero_pop_cbgs:,}")

print(f"\n--- HH Share Distribution ---")
print(cb_df['hh_share_in_cbg'].describe())

Rolled-up CBG totals: 77,659 CBGs

HH share validation — non-zero CBGs: 77,049 | Failed sum-to-1: 0
Pop share validation — non-zero CBGs: 77,182 | Failed sum-to-1: 0

Zero-HH CBGs (all CBs have 0 occupied HU): 610
Zero-Pop CBGs (all CBs have 0 population): 477

--- HH Share Distribution ---
count   2,079,994.00
mean            0.04
std             0.08
min             0.00
25%             0.00
50%             0.01
75%             0.04
max             1.00
Name: hh_share_in_cbg, dtype: float64


## 7. Compute Occupancy Rate & Allocate Projections

**Occupancy rate** converts Housing Units (HU) to Households (HH):
```
occupancy_rate = CBG_2020_HH_Count / CBG_2020_HU_Count    (from Decennial Census)
```

**HH projection rolldown** (three-step: project → allocate → convert):
```
projected_hh_cb = CBG_HU_2024_ACS × hh_share × occupancy_rate
```

**Population projection rolldown** (two-step: project → allocate):
```
projected_pop_cb = CBG_Pop_2024_ACS × pop_share
```

In [7]:
### Join CBG Decennial Housing for occupancy rate
cb_df = cb_df.merge(
    cbg_housing_dec[['cbg_fips', 'cbg_total_hu_2020', 'cbg_hh_2020']],
    on='cbg_fips', how='left'
)

### Occupancy rate: occupied HU / total HU (from 2020 Decennial Census at CBG level)
cb_df['occupancy_rate'] = np.where(
    cb_df['cbg_total_hu_2020'] > 0,
    cb_df['cbg_hh_2020'] / cb_df['cbg_total_hu_2020'],
    0.0
)

print("--- Occupancy Rate Distribution (non-zero only) ---")
occ_valid = cb_df.loc[cb_df['occupancy_rate'] > 0, 'occupancy_rate']
print(occ_valid.describe())

### Join CBG-level ACS 2024 projections
cb_df = cb_df.merge(cbg_hu_acs_2024[['cbg_fips', 'cbg_hu_proj_2024']], on='cbg_fips', how='left')
cb_df = cb_df.merge(cbg_pop_acs_2024[['cbg_fips', 'cbg_pop_proj_2024']], on='cbg_fips', how='left')

### HH Projection Rolldown: CBG_HU_2024 × hh_share × occupancy_rate
cb_df['projected_hh'] = (
    cb_df['cbg_hu_proj_2024'] * cb_df['hh_share_in_cbg'] * cb_df['occupancy_rate']
)

### Population Projection Rolldown: CBG_Pop_2024 × pop_share
cb_df['projected_pop'] = (
    cb_df['cbg_pop_proj_2024'] * cb_df['pop_share_in_cbg']
)

### HH Growth Rate
cb_df['hh_growth_rate'] = np.where(
    cb_df['occupied_housing_units'] > 0,
    (cb_df['projected_hh'] - cb_df['occupied_housing_units']) / cb_df['occupied_housing_units'],
    np.nan
)

print(f"\n--- Projected HH Distribution ---")
print(cb_df['projected_hh'].describe())
print(f"\n--- Projected Pop Distribution ---")
print(cb_df['projected_pop'].describe())
print(f"\n--- HH Growth Rate Distribution ---")
print(cb_df['hh_growth_rate'].describe())

--- Occupancy Rate Distribution (non-zero only) ---
count   2,075,624.00
mean            0.89
std             0.11
min             0.01
25%             0.86
50%             0.92
75%             0.96
max             1.00
Name: occupancy_rate, dtype: float64

--- Projected HH Distribution ---
count   2,079,193.00
mean           20.04
std            49.75
min             0.00
25%             0.00
50%             5.81
75%            20.75
max         2,868.53
Name: projected_hh, dtype: float64

--- Projected Pop Distribution ---
count   2,079,193.00
mean           54.29
std           124.23
min             0.00
25%             0.00
50%            17.77
75%            58.71
max         8,826.35
Name: projected_pop, dtype: float64

--- HH Growth Rate Distribution ---
count   1,419,417.00
mean            0.02
std             0.41
min            -1.00
25%            -0.11
50%             0.00
75%             0.13
max           207.00
Name: hh_growth_rate, dtype: float64


## 8. Join Area Data & Compute Densities

Load the `area_sq_km.csv` produced by `01_shapefile_processing.ipynb` and compute housing and population density at the CB level.

```
housing_density = occupied_housing_units / area_sq_km
pop_density     = total_population / area_sq_km
```

Zero-area blocks (water-only) get `NaN` density to avoid division by zero.

In [8]:
### Load area data from Phase 1
area_df = pd.read_csv(AREA_CSV, dtype={'cb_fips': str})
print(f"Area data loaded: {len(area_df):,} blocks")

### Merge area onto CB demographics
cb_df = cb_df.merge(area_df, on='cb_fips', how='left')

area_matched = cb_df['area_sq_km'].notna().sum()
print(f"Area matched: {area_matched:,} / {len(cb_df):,} ({area_matched/len(cb_df)*100:.1f}%)")

### Compute densities (NaN for zero-area blocks)
cb_df['housing_density'] = np.where(
    cb_df['area_sq_km'] > 0,
    cb_df['occupied_housing_units'] / cb_df['area_sq_km'],
    np.nan
)

cb_df['pop_density'] = np.where(
    cb_df['area_sq_km'] > 0,
    cb_df['total_population'] / cb_df['area_sq_km'],
    np.nan
)

print(f"\n--- Housing Density (HH/sq km) ---")
print(cb_df['housing_density'].describe())
print(f"\n--- Population Density (pop/sq km) ---")
print(cb_df['pop_density'].describe())

Area data loaded: 2,079,994 blocks
Area matched: 2,079,994 / 2,079,994 (100.0%)

--- Housing Density (HH/sq km) ---
count   2,079,994.00
mean          400.41
std         1,190.76
min             0.00
25%             0.00
50%            53.33
75%           503.60
max       462,081.14
Name: housing_density, dtype: float64

--- Population Density (pop/sq km) ---
count   2,079,994.00
mean        1,153.98
std         2,799.33
min             0.00
25%             0.00
50%           224.40
75%         1,502.01
max       924,162.28
Name: pop_density, dtype: float64


## 9. Quality Checks & Validation

In [ ]:
print("=" * 60)
print("QUALITY CHECK REPORT")
print("=" * 60)

# 1. CB FIPS integrity
null_cb = cb_df['cb_fips'].isna().sum()
bad_len = (cb_df['cb_fips'].str.len() != 15).sum()
print(f"\n[{'PASS' if null_cb == 0 else 'FAIL'}] Null cb_fips: {null_cb}")
print(f"[{'PASS' if bad_len == 0 else 'FAIL'}] Invalid cb_fips length (!=15): {bad_len}")

# 2. CBG derivation
null_cbg = cb_df['cbg_fips'].isna().sum()
print(f"[{'PASS' if null_cbg == 0 else 'FAIL'}] Null cbg_fips: {null_cbg}")

# 3. MHI coverage
for year, col in [('2020', 'mhi_2020'), ('2024', 'mhi_2024')]:
    n_null = cb_df[col].isna().sum()
    pct = n_null / len(cb_df) * 100
    status = 'PASS' if pct < 10 else 'WARN'
    print(f"[{status}] MHI {year} missing: {n_null:,} ({pct:.1f}%)")

# 4. Share sums
hh_sums = cb_df.groupby('cbg_fips')['hh_share_in_cbg'].sum()
hh_bad = ((hh_sums[hh_sums > 0] - 1.0).abs() > 1e-6).sum()
print(f"[{'PASS' if hh_bad == 0 else 'FAIL'}] HH shares not summing to 1.0: {hh_bad} CBGs")

pop_sums = cb_df.groupby('cbg_fips')['pop_share_in_cbg'].sum()
pop_bad = ((pop_sums[pop_sums > 0] - 1.0).abs() > 1e-6).sum()
print(f"[{'PASS' if pop_bad == 0 else 'FAIL'}] Pop shares not summing to 1.0: {pop_bad} CBGs")

# 5. Occupancy rate range (typical: 0.85-0.95)
extreme_occ = cb_df.loc[
    (cb_df['occupancy_rate'] > 0) &
    ((cb_df['occupancy_rate'] < 0.5) | (cb_df['occupancy_rate'] > 1.0))
]
print(f"[{'PASS' if len(extreme_occ) == 0 else 'INFO'}] CBs with extreme occupancy rate (<0.5 or >1.0): {len(extreme_occ):,}")

# 6. Negative/extreme MHI
neg_mhi = (cb_df['mhi_2024'].dropna() < 0).sum()
high_mhi = (cb_df['mhi_2024'].dropna() > 500000).sum()
print(f"[{'PASS' if neg_mhi == 0 else 'WARN'}] Negative MHI 2024: {neg_mhi}")
print(f"[INFO] MHI 2024 > $500K: {high_mhi:,}")

# 7. Area & density coverage
area_null = cb_df['area_sq_km'].isna().sum()
area_pct = area_null / len(cb_df) * 100
print(f"[{'PASS' if area_pct < 1 else 'WARN'}] Area missing: {area_null:,} ({area_pct:.1f}%)")
density_null = cb_df['housing_density'].isna().sum()
print(f"[INFO] Density NaN (zero-area blocks): {density_null:,}")

# 8. Summary
print(f"\n{'=' * 60}")
print(f"SUMMARY")
print(f"{'=' * 60}")
print(f"Total Census Blocks:     {len(cb_df):,}")
print(f"Unique CBGs (6 states):  {cb_df['cbg_fips'].nunique():,}")
print(f"States:                  {sorted(cb_df['cb_fips'].str[:2].unique())}")

for st_fips, st_name in sorted(STATE_NAMES.items()):
    n = (cb_df['cb_fips'].str[:2] == st_fips).sum()
    print(f"  {st_name} ({st_fips}): {n:,} CBs")

print(f"\nCBs with zero HH:       {(cb_df['occupied_housing_units'] == 0).sum():,}")
print(f"CBs with zero pop:       {(cb_df['total_population'] == 0).sum():,}")

print(f"\n--- MHI 2024 Distribution ---")
print(cb_df['mhi_2024'].describe())

print(f"\n--- Occupancy Rate Distribution (all CBs) ---")
print(cb_df['occupancy_rate'].describe())

## 10. Export

Export the complete CB-level demographics table with all 19 columns as Parquet and CSV.

In [10]:
### Select final output columns
output_cols = [
    'cb_fips', 'cbg_fips',
    'total_housing_units', 'occupied_housing_units', 'vacant_housing_units',
    'total_population',
    'mhi_2020', 'mhi_moe_2020', 'mhi_2024', 'mhi_moe_2024',
    'hh_share_in_cbg', 'pop_share_in_cbg',
    'occupancy_rate',
    'projected_hh', 'hh_growth_rate', 'projected_pop',
    'area_sq_km', 'housing_density', 'pop_density'
]

output_df = cb_df[output_cols].copy()

### Export Parquet
parquet_path = os.path.join(OUTPUT_DIR, 'cb_demographics.parquet')
output_df.to_parquet(parquet_path, index=False, engine='pyarrow')

### Export CSV (for inspection)
csv_path = os.path.join(OUTPUT_DIR, 'cb_demographics.csv')
output_df.to_csv(csv_path, index=False)

print(f"Parquet: {parquet_path}")
print(f"CSV:     {csv_path}")
print(f"Rows:    {len(output_df):,}")
print(f"Columns: {output_df.shape[1]}")

parquet_mb = os.path.getsize(parquet_path) / (1024 * 1024)
csv_mb = os.path.getsize(csv_path) / (1024 * 1024)
print(f"Parquet size: {parquet_mb:.1f} MB")
print(f"CSV size:     {csv_mb:.1f} MB")

print(f"\nColumn dtypes:")
print(output_df.dtypes)
print(f"\nFirst 5 rows:")
output_df.head()

Parquet: ../../output/cb_demographics.parquet
CSV:     ../../output/cb_demographics.csv
Rows:    2,079,994
Columns: 19
Parquet size: 98.6 MB
CSV size:     410.9 MB

Column dtypes:
cb_fips                    object
cbg_fips                   object
total_housing_units         int64
occupied_housing_units      int64
vacant_housing_units        int64
total_population            int64
mhi_2020                  float64
mhi_moe_2020              float64
mhi_2024                  float64
mhi_moe_2024              float64
hh_share_in_cbg           float64
pop_share_in_cbg          float64
occupancy_rate            float64
projected_hh              float64
hh_growth_rate            float64
projected_pop             float64
area_sq_km                float64
housing_density           float64
pop_density               float64
dtype: object

First 5 rows:


,cb_fips,cbg_fips,total_housing_units,occupied_housing_units,vacant_housing_units,total_population,mhi_2020,mhi_moe_2020,mhi_2024,mhi_moe_2024,hh_share_in_cbg,pop_share_in_cbg,occupancy_rate,projected_hh,hh_growth_rate,projected_pop,area_sq_km,housing_density,pop_density
0,060014001001000,060014001001,0,0,0,0,"214,500.00","86,478.00",NaN,NaN,0.00,0.00,0.93,0.00,NaN,0.00,0.08,0.00,0.00
1,060014001001001,060014001001,0,0,0,0,"214,500.00","86,478.00",NaN,NaN,0.00,0.00,0.93,0.00,NaN,0.00,1.12,0.00,0.00
2,060014001001002,060014001001,0,0,0,0,"214,500.00","86,478.00",NaN,NaN,0.00,0.00,0.93,0.00,NaN,0.00,0.02,0.00,0.00
3,060014001001003,060014001001,0,0,0,0,"214,500.00","86,478.00",NaN,NaN,0.00,0.00,0.93,0.00,NaN,0.00,1.98,0.00,0.00
4,060014001001004,060014001001,0,0,0,0,"214,500.00","86,478.00",NaN,NaN,0.00,0.00,0.93,0.00,NaN,0.00,0.01,0.00,0.00
